# Notebook 03: Layer 1 and Layer 2 Evaluation

Evaluates the exact-match and embedding-similarity detection layers:
- Layer 1 (Aho-Corasick) precision, recall, F1, AUROC on all corpora
- Layer 2 (SPECTER + context coherence) evaluation
- Threshold calibration for Layer 2
- Variant and paraphrase detection examples

**Prerequisites:**
- `notebooks/01_data_acquisition.ipynb` must have run first
  (produces `data/ground_truth/confirmed_tortured.tsv` and `confirmed_clean.tsv`)
- Layer 1: no ML models required
- Layer 2: requires `sentence-transformers` and SPECTER model (~400 MB download on first run)

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from tpc.registry.loader import load_registry
from tpc.layers.exact_match import ExactMatchDetector
from tpc.evaluation.metrics import evaluate_layer_on_corpus, LayerMetrics

DATA_DIR     = ROOT / 'data' / 'ground_truth'
APPENDIX_DIR = ROOT / 'data' / 'appendix'
FIGURES_DIR  = ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## 1. Load evaluation corpora

In [ ]:
# Load corpora produced by notebook 01
# If 01 has not been run yet, fall back to a small synthetic corpus for smoke-testing

tortured_path = DATA_DIR / 'confirmed_tortured.tsv'
clean_path    = DATA_DIR / 'confirmed_clean.tsv'

if tortured_path.exists() and clean_path.exists():
    df_pos = pd.read_csv(tortured_path, sep='\t')
    df_neg = pd.read_csv(clean_path,    sep='\t')
    print(f'Loaded PubMed corpora: {len(df_pos)} retracted, {len(df_neg)} clean')
else:
    print('PubMed corpora not found — using synthetic smoke-test corpus.')
    print('Run notebooks/01_data_acquisition.ipynb first for real results.')
    # Synthetic fallback
    tortured_examples = [
        'We examined the amino corrosive sequence in detail.',
        'The profound learning model outperformed baselines.',
        'Signal to noise was computed as flag to commotion ratio.',
        'Bosom peril rates were elevated in the study population.',
        'Irregular examination of 200 subjects was performed.',
        'The fake neural organization achieved 94% accuracy.',
        'Nucleic harsh binding was measured spectrophotometrically.',
    ] * 30
    clean_examples = [
        'We examined the amino acid sequence in detail.',
        'The deep learning model outperformed baselines.',
        'Signal to noise ratio was computed for each condition.',
        'Breast cancer rates were elevated in the study population.',
        'Random sampling of 200 subjects was performed.',
        'The artificial neural network achieved 94% accuracy.',
        'Nucleic acid binding was measured spectrophotometrically.',
    ] * 30
    df_pos = pd.DataFrame({'abstract': tortured_examples, 'label': 'retracted'})
    df_neg = pd.DataFrame({'abstract': clean_examples,    'label': 'clean'})
    print(f'Synthetic corpus: {len(df_pos)} retracted, {len(df_neg)} clean')

papers_all = df_pos.to_dict('records') + df_neg.to_dict('records')
print(f'Total evaluation corpus: {len(papers_all)} papers')

## 2. Layer 1 Evaluation

In [ ]:
signals = load_registry(signals_dir=ROOT / 'signals' / 'phrases')
l1 = ExactMatchDetector(signals=signals)

metrics_l1 = evaluate_layer_on_corpus(
    detector=l1,
    papers=papers_all,
    layer_name='L1_exact_match',
    corpus_name='positive+negative',
)

print('=== Layer 1 Results ===')
d = metrics_l1.to_dict()
for k, v in d.items():
    if k not in ('layer', 'corpus'):
        print(f'  {k:25s}: {v}')

In [ ]:
# False positive analysis — which clean papers does L1 incorrectly flag?
fp_examples = []
for paper in df_neg.to_dict('records')[:200]:
    hits = l1.detect(paper.get('abstract', ''))
    if hits:
        fp_examples.append({
            'abstract_snippet': paper['abstract'][:120],
            'hit_tortured':     hits[0]['tortured'],
            'hit_canonical':    hits[0]['canonical'],
            'context':          hits[0].get('context', '')[:80],
        })

print(f'Layer 1 false positives on first 200 clean papers: {len(fp_examples)}')
if fp_examples:
    print('Examples:')
    for ex in fp_examples[:3]:
        print(' ', ex)

## 3. Layer 1 — Variant coverage analysis

In [ ]:
# For each confirmed signal, show how many known_variants are registered
# and test that all variants are caught
import yaml

variant_results = []
for sig in signals:
    test_texts = [f'This paper discusses {v} in detail.' for v in sig.all_terms]
    caught = 0
    for text in test_texts:
        hits = l1.detect(text)
        if hits:
            caught += 1
    variant_results.append({
        'signal_id':      sig.id,
        'tortured':       sig.tortured,
        'total_variants': len(sig.all_terms),
        'variants_caught': caught,
        'variant_recall':  caught / len(sig.all_terms),
    })

df_variants = pd.DataFrame(variant_results)
print('=== Layer 1 Variant Coverage ===')
print(df_variants.to_string(index=False))
print(f'\nMean variant recall: {df_variants["variant_recall"].mean():.3f}')

## 4. Layer 2 Evaluation (SPECTER embeddings)

> **Note:** First run downloads the SPECTER model (~400 MB). Allow 5-10 min on first run.

In [ ]:
# Uncomment to run Layer 2 (requires sentence-transformers)
# from tpc.layers.embedding import EmbeddingDetector
#
# print('Initializing Layer 2 (downloading SPECTER if not cached)...')
# l2 = EmbeddingDetector(signals=signals)
# print('Layer 2 ready.')
#
# metrics_l2 = evaluate_layer_on_corpus(
#     detector=l2,
#     papers=papers_all[:200],  # subset for speed
#     layer_name='L2_embedding',
#     corpus_name='positive+negative',
# )
# print('=== Layer 2 Results ===')
# for k, v in metrics_l2.to_dict().items():
#     if k not in ('layer', 'corpus'):
#         print(f'  {k:25s}: {v}')

print('Layer 2 cell is commented out. Uncomment and run to evaluate.')
print('Full Layer 2 evaluation is included in notebook 05 (combined ablation).')

## 5. Layer 2 — Threshold calibration

In [ ]:
# This cell demonstrates how the Layer 2 thresholds were calibrated.
# Requires SPECTER. Uncomment to run.

# from tpc.layers.embedding import EmbeddingDetector
# from sklearn.metrics import precision_score, recall_score
#
# suspicion_thresholds = [0.20, 0.25, 0.30, 0.35, 0.40]
# sim_thresholds       = [0.75, 0.80, 0.82, 0.85, 0.90]
# calibration_papers   = papers_all[:100]  # use first 100 for speed
#
# results = []
# for sim_t in sim_thresholds:
#     for sus_t in suspicion_thresholds:
#         det = EmbeddingDetector(
#             signals=signals,
#             sim_threshold=sim_t,
#             suspicion_threshold=sus_t
#         )
#         m = evaluate_layer_on_corpus(det, calibration_papers, 'L2', 'calib')
#         results.append({'sim_t': sim_t, 'sus_t': sus_t,
#                         'precision': m.precision, 'recall': m.recall, 'f1': m.f1})
#
# df_calib = pd.DataFrame(results)
# # Best config satisfying precision >= 0.95:
# valid = df_calib[df_calib['precision'] >= 0.95]
# best  = valid.loc[valid['recall'].idxmax()]
# print('Best threshold config (precision>=0.95, max recall):')
# print(best)

print('Threshold calibration cell is commented out. Uncomment to run.')
print('Selected thresholds: sim_threshold=0.82, suspicion_threshold=0.30')

## 6. Precision-Recall curve for Layer 1

In [ ]:
from sklearn.metrics import precision_recall_curve, auc

scores = metrics_l1.scores
labels = metrics_l1.labels

if len(set(labels)) == 2:
    precision_pts, recall_pts, _ = precision_recall_curve(labels, scores)
    pr_auc = auc(recall_pts, precision_pts)

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(recall_pts, precision_pts, color='#2166ac', lw=2,
            label=f'L1 exact match (AUC = {pr_auc:.3f})')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall Curve: Layer 1 (Exact Match)')
    ax.legend()
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_pr_l1.pdf', dpi=300, bbox_inches='tight')
    plt.show()
    print(f'PR-AUC (L1): {pr_auc:.4f}')
else:
    print('Only one class in labels — cannot compute PR curve. Use real PubMed corpus.')